# Analisis exploratorio: operadores de vecindad en MetaCARP

**Corpus**: `resultados_consolidados.csv` (92,529 corridas de 5 metaheuristicas
sobre 23 instancias pequenas: gdb1-7, gdb10, gdb12-17, gdb19-21, kshs1-6).

**Pregunta motora**: de los 9 operadores de vecindad disponibles
(3 intra-ruta + 6 inter-ruta), cuales producen mejoras de forma consistente
a traves de instancias y metaheuristicas, y cuales son candidatos a poda?

**Las metricas clave por operador** (cada una repetida para los 9 ops):
- `propuesto_{op}`: veces que la metaheuristica intento aplicar el operador
- `aceptado_{op}`: veces que el vecino generado fue aceptado como solucion actual
- `mejoraron_{op}`: veces que el operador produjo una mejora sobre la solucion actual
- `trayectoria_mejor_{op}`: veces que el operador participo en la trayectoria del mejor global

Este notebook NO toma decisiones de poda todavia, solo genera la evidencia.

## 1. Setup y carga de datos

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuracion visual: paleta limpia y tamano consistente
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

# Mostrar mas filas/cols en pandas
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

In [ ]:
RUTA_CSV = Path('resultados_consolidados.csv')
df = pd.read_csv(RUTA_CSV, low_memory=False)
print(f'Filas:    {len(df):,}')
print(f'Columnas: {len(df.columns)}')
df.head(3)

In [ ]:
# Los 9 operadores agrupados por categoria (intra vs inter ruta)
OPS_INTRA = ['relocate_intra', 'swap_intra', '2opt_intra']
OPS_INTER = ['relocate_inter', 'swap_inter', '2opt_star',
             'cross_exchange', 'or_opt_2', 'or_opt_3']
OPS = OPS_INTRA + OPS_INTER

# Helpers para acceder a las 4 perspectivas
def cols(prefijo): return [f'{prefijo}_{op}' for op in OPS]
COLS_PROPUESTO  = cols('propuesto')
COLS_ACEPTADO   = cols('aceptado')
COLS_MEJORARON  = cols('mejoraron')
COLS_TRAY_MEJOR = cols('trayectoria_mejor')

## 2. Resumen general del corpus

Verificamos volumen por metaheuristica, instancias presentes y BKS de referencia.

In [ ]:
# Conteo de corridas por metaheuristica
resumen_mh = (df.groupby('metaheuristica')
                .agg(n_corridas=('mejor_costo', 'size'),
                     instancias=('instancia', 'nunique'),
                     tiempo_total_s=('tiempo_segundos', 'sum'),
                     tiempo_mediano_s=('tiempo_segundos', 'median'))
                .sort_values('n_corridas', ascending=False))
resumen_mh

In [ ]:
# Cobertura instancia x MH (deberia ser homogenea: ~mismo numero de combos por celda)
cobertura = df.groupby(['metaheuristica', 'instancia']).size().unstack(fill_value=0)
print('Forma:', cobertura.shape)
cobertura

In [ ]:
# BKS de referencia por instancia (deberia ser unico)
bks = (df.groupby('instancia')
         .agg(bks=('bks_referencia', 'first'),
              n_distintos=('bks_referencia', 'nunique'))
         .sort_index())
assert bks['n_distintos'].max() == 1, 'BKS inconsistente entre corridas de la misma instancia'
bks.drop(columns='n_distintos')

## 3. Calidad de soluciones por metaheuristica

Antes de mirar operadores, contextualizamos cuanto tan bien funciona cada MH.
El operador mas usado por una MH mediocre puede no ser util en una MH ganadora.

In [ ]:
# Estadisticas de gap respecto al BKS (menor = mejor)
calidad = (df.groupby('metaheuristica')['gap_bks_porcentaje']
             .agg(['mean', 'median', 'std', 'min', 'max'])
             .round(3))
calidad

In [ ]:
# Tasa de factibilidad del mejor global encontrado (debe ser 1.0 o muy cercana)
(df.groupby('metaheuristica')['mejor_solucion_factible_final']
   .mean()
   .rename('tasa_factibilidad').round(4))

In [ ]:
# Distribucion del gap por MH para ver dispersion
fig, ax = plt.subplots(figsize=(10, 5))
orden = (df.groupby('metaheuristica')['gap_bks_porcentaje']
           .median().sort_values().index)
sns.boxplot(data=df, x='metaheuristica', y='gap_bks_porcentaje',
            order=orden, ax=ax, showfliers=False)
ax.set_title('Gap respecto al BKS por metaheuristica')
ax.set_xlabel('')
ax.set_ylabel('gap (%)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 4. Analisis del uso de operadores

Las cuatro perspectivas (`propuesto`, `aceptado`, `mejoraron`, `trayectoria_mejor`)
miden distintas dimensiones del valor de un operador:

- **Propuesto**: cuanto lo usa la politica (sesgo inter/intra + uniforme dentro del grupo)
- **Tasa de aceptacion** = aceptado / propuesto: que tan facil es que el vecino sea adoptado
- **Tasa de mejora**    = mejoraron / propuesto: cuantas veces produjo una solucion mejor que la actual
- **Contribucion a mejor** = trayectoria_mejor / propuesto: cuanto contribuyo a refinar el mejor global

Notese que un operador con alta aceptacion pero baja mejora solo esta moviendo lateral.

### 4.1 Sumas globales por operador

In [ ]:
# Totales agregados a traves de TODAS las corridas (todas MH, todas instancias, todas reps)
totales = pd.DataFrame({
    'propuesto':         df[COLS_PROPUESTO].sum().values,
    'aceptado':          df[COLS_ACEPTADO].sum().values,
    'mejoraron':         df[COLS_MEJORARON].sum().values,
    'trayectoria_mejor': df[COLS_TRAY_MEJOR].sum().values,
}, index=OPS)
totales['tasa_aceptacion'] = totales['aceptado']  / totales['propuesto']
totales['tasa_mejora']     = totales['mejoraron'] / totales['propuesto']
totales['contrib_mejor']   = totales['trayectoria_mejor'] / totales['propuesto']
totales['grupo'] = ['intra']*3 + ['inter']*6
totales.round(4)

In [ ]:
# Visualizacion: propuestas vs mejoras absolutas por operador
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colores = ['#1f77b4' if g == 'intra' else '#ff7f0e' for g in totales['grupo']]

axes[0].bar(totales.index, totales['propuesto'], color=colores)
axes[0].set_title('Total propuestas por operador (escala log)')
axes[0].set_yscale('log')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_ylabel('# propuestas (log)')

axes[1].bar(totales.index, totales['mejoraron'], color=colores)
axes[1].set_title('Total mejoras producidas por operador')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylabel('# mejoras')

from matplotlib.patches import Patch
handles = [Patch(color='#1f77b4', label='intra'), Patch(color='#ff7f0e', label='inter')]
fig.legend(handles=handles, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.02))
plt.tight_layout()
plt.show()

In [ ]:
# Tasa de aceptacion y de mejora (en %) por operador, ordenadas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tasas_ord = totales.sort_values('tasa_mejora', ascending=True)

axes[0].barh(tasas_ord.index, tasas_ord['tasa_aceptacion'] * 100,
             color=['#1f77b4' if g == 'intra' else '#ff7f0e' for g in tasas_ord['grupo']])
axes[0].set_title('Tasa de aceptacion (%) por operador')
axes[0].set_xlabel('aceptado / propuesto (%)')

axes[1].barh(tasas_ord.index, tasas_ord['tasa_mejora'] * 100,
             color=['#1f77b4' if g == 'intra' else '#ff7f0e' for g in tasas_ord['grupo']])
axes[1].set_title('Tasa de mejora (%) por operador')
axes[1].set_xlabel('mejoraron / propuesto (%)')

plt.tight_layout()
plt.show()

### 4.2 Tasa de mejora por metaheuristica x operador

In [ ]:
# Esta es la vista clave: la fila de cada MH muestra cuanta mejora obtuvo de cada operador.
# Un operador con celdas oscuras en varias MH es valioso transversalmente.

def agg_por_mh(df_in, cols_num, cols_den):
    '''Suma numerador y denominador por MH y devuelve el ratio. Evita NaN cuando den=0.'''
    num = df_in.groupby('metaheuristica')[cols_num].sum()
    den = df_in.groupby('metaheuristica')[cols_den].sum()
    num.columns = OPS
    den.columns = OPS
    return (num / den.replace(0, np.nan)).fillna(0)

tasa_mejora_mh = agg_por_mh(df, COLS_MEJORARON, COLS_PROPUESTO)
tasa_acept_mh  = agg_por_mh(df, COLS_ACEPTADO,  COLS_PROPUESTO)

fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(tasa_mejora_mh * 100, annot=True, fmt='.2f', cmap='YlGnBu',
            cbar_kws={'label': 'tasa mejora (%)'}, ax=ax)
ax.set_title('Tasa de mejora (%) por metaheuristica x operador')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(tasa_acept_mh * 100, annot=True, fmt='.2f', cmap='Purples',
            cbar_kws={'label': 'tasa aceptacion (%)'}, ax=ax)
ax.set_title('Tasa de aceptacion (%) por metaheuristica x operador')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

### 4.3 Contribucion a la trayectoria del mejor global

In [ ]:
# trayectoria_mejor: cuantas veces el operador estuvo en el camino al mejor global.
# Es la metrica mas exigente: no basta con producir vecinos mejores que el actual,
# tiene que aportar al optimo global encontrado.

contrib_mh = agg_por_mh(df, COLS_TRAY_MEJOR, COLS_PROPUESTO)
fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(contrib_mh * 100, annot=True, fmt='.3f', cmap='OrRd',
            cbar_kws={'label': 'contribucion a mejor (%)'}, ax=ax)
ax.set_title('Contribucion a la trayectoria del mejor (%) por MH x operador')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

### 4.4 Variabilidad por instancia

In [ ]:
# Tasa de mejora promedio (sobre todas las corridas) para cada par operador-instancia.
# Si un operador es valioso solo en 2-3 instancias y nulo en el resto, es candidato a
# operador especializado y no general.

num_inst = df.groupby('instancia')[COLS_MEJORARON].sum()
den_inst = df.groupby('instancia')[COLS_PROPUESTO].sum()
num_inst.columns = OPS
den_inst.columns = OPS
tasa_mejora_inst = (num_inst / den_inst.replace(0, np.nan)).fillna(0)

fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(tasa_mejora_inst * 100, annot=True, fmt='.2f', cmap='YlGnBu',
            cbar_kws={'label': 'tasa mejora (%)'}, ax=ax)
ax.set_title('Tasa de mejora (%) por instancia x operador')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 5. Operadores intra vs inter

Al sumar los tres operadores intra y los seis inter, comparamos el peso relativo de cada grupo.

In [ ]:
intra_inter = pd.DataFrame({
    'intra': df[[f'mejoraron_{op}' for op in OPS_INTRA]].sum(axis=1),
    'inter': df[[f'mejoraron_{op}' for op in OPS_INTER]].sum(axis=1),
})
intra_inter['metaheuristica'] = df['metaheuristica']

# Promedio de mejoras absolutas por corrida, separado por grupo
g = intra_inter.groupby('metaheuristica')[['intra', 'inter']].sum()
g['fraccion_inter'] = g['inter'] / (g['intra'] + g['inter'])
g.round(3)

In [ ]:
# Visualizacion stacked: composicion intra/inter en mejoras totales por MH
fig, ax = plt.subplots(figsize=(10, 4))
g[['intra', 'inter']].plot(kind='barh', stacked=True, ax=ax,
                            color=['#1f77b4', '#ff7f0e'])
ax.set_title('Composicion de mejoras intra vs inter por metaheuristica (totales absolutos)')
ax.set_xlabel('# mejoras totales')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 6. Candidatos a poda: criterio multidimensional

Un operador es candidato a poda si cumple **simultaneamente**:

1. **Baja tasa de mejora** (mejoras / propuestas) en mas de 3 metaheuristicas
2. **Baja contribucion** a la trayectoria del mejor en mas de 3 MH
3. **Cobertura limitada de instancias** (solo aporta en pocas)

Los umbrales son orientativos; el objetivo aqui es generar la evidencia, no decidir.

In [ ]:
# Construimos una tabla resumen por operador con las metricas clave consolidadas

resumen_op = pd.DataFrame(index=OPS)
resumen_op['grupo'] = ['intra']*3 + ['inter']*6
resumen_op['propuestas_totales']         = totales['propuesto']
resumen_op['mejoras_totales']            = totales['mejoraron']
resumen_op['tasa_mejora_global_%']       = totales['tasa_mejora'] * 100
resumen_op['contrib_mejor_global_%']     = totales['contrib_mejor'] * 100
resumen_op['mh_con_tasa_mejora_>1%']     = (tasa_mejora_mh > 0.01).sum()
resumen_op['mh_con_contrib_>0.5%']       = (contrib_mh > 0.005).sum()
resumen_op['instancias_con_mejora_>0']   = (tasa_mejora_inst > 0).sum()
resumen_op.sort_values('tasa_mejora_global_%', ascending=False).round(4)

In [ ]:
# Ranking final de operadores: combinamos las metricas en un score 0-1 simple,
# normalizando cada columna a [0,1] y promediando. Sirve solo para ordenar.

metricas_score = ['tasa_mejora_global_%', 'contrib_mejor_global_%',
                   'mh_con_tasa_mejora_>1%', 'mh_con_contrib_>0.5%',
                   'instancias_con_mejora_>0']
norm = resumen_op[metricas_score].copy()
for c in metricas_score:
    rango = norm[c].max() - norm[c].min()
    norm[c] = (norm[c] - norm[c].min()) / rango if rango > 0 else 0.0
resumen_op['score_promedio'] = norm.mean(axis=1).round(3)

ranking = resumen_op.sort_values('score_promedio', ascending=False)
ranking[['grupo', 'tasa_mejora_global_%', 'contrib_mejor_global_%',
         'mh_con_tasa_mejora_>1%', 'instancias_con_mejora_>0', 'score_promedio']]

In [ ]:
# Visualizacion del ranking
fig, ax = plt.subplots(figsize=(10, 5))
colores_rank = ['#1f77b4' if g == 'intra' else '#ff7f0e'
                for g in ranking['grupo']]
ax.barh(ranking.index, ranking['score_promedio'], color=colores_rank)
ax.set_xlabel('score promedio (0-1)')
ax.set_title('Ranking de operadores por score combinado')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Notas para la siguiente fase

Este notebook deja sobre la mesa la evidencia necesaria para decidir:

- La tabla `ranking` ordena los 9 operadores por un score combinado
- Los heatmaps muestran si un operador con score bajo a nivel global es
  valioso para alguna MH o instancia especifica
- La comparacion intra/inter responde si conviene podar dentro de cada grupo
  manteniendo el balance del helper `seleccionar_grupo_operadores_inter_intra`

Decisiones que **NO** se toman aqui (van en la siguiente fase):
- Umbral exacto de poda
- Como modificar `OPERADORES_POPULARES` en `vecindarios.py`
- Re-correr el grid con el subconjunto reducido para confirmar que no degrada calidad